# مختبر تعديل البيانات: INSERT, UPDATE, DELETE
سنطبق أوامر SQL لتعديل البيانات على قاعدة بيانات SQLite ضمن الذاكرة.

## إعداد قاعدة البيانات والبيانات الأولية
سنقوم باستيراد المكتبات اللازمة، وإنشاء اتصال بقاعدة بيانات داخل الذاكرة، وإنشاء الجداول، وإدراج بيانات تجريبية.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
np.random.seed(42)
# إنشاء اتصال بقاعدة بيانات داخل الذاكرة
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
print('تم إنشاء قاعدة بيانات في الذاكرة.')

# إنشاء الجداول
cursor.execute('''CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    stock INTEGER NOT NULL
)''')
cursor.execute('''CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    join_date TEXT NOT NULL
)''')
cursor.execute('''CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(id),
    FOREIGN KEY (product_id) REFERENCES products(id)
)''')
conn.commit()
print('تم إنشاء الجداول.')

# إدراج بيانات أولية
# بيانات المنتجات
product_names = ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Printer']
categories = ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics']
prices = np.random.uniform(50, 1000, size=5).round(2)
stock = np.random.randint(10, 100, size=5)
products_data = list(zip(product_names, categories, prices, stock))
cursor.executemany('INSERT INTO products (name, category, price, stock) VALUES (?, ?, ?, ?)', products_data)

# بيانات العملاء
customer_names = ['Ali', 'Sara', 'Omar', 'Laila']
customer_emails = ['ali@example.com', 'sara@example.com', 'omar@example.com', 'laila@example.com']
join_dates = ['2022-01-15', '2022-03-20', '2022-06-10', '2022-09-25']
customers_data = list(zip(customer_names, customer_emails, join_dates))
cursor.executemany('INSERT INTO customers (name, email, join_date) VALUES (?, ?, ?)', customers_data)

# بيانات الطلبات
dates = ['2022-01-20', '2022-04-01', '2022-07-05', '2022-10-15', '2023-01-10', '2023-04-15']
orders = []
for i, date in enumerate(dates):
    c_id = np.random.randint(1, 5)
    p_id = np.random.randint(1, 6)
    qty = np.random.randint(1, 5)
    orders.append((c_id, p_id, qty, date))
cursor.executemany('INSERT INTO orders (customer_id, product_id, quantity, order_date) VALUES (?, ?, ?, ?)', orders)
conn.commit()
print('تم إدراج البيانات الأولية.')

## عرض البيانات الأولية
لنستعرض الجداول بعد الإعداد.

In [ ]:
# عرض جدول المنتجات
df_products = pd.read_sql_query('SELECT * FROM products', conn)
print('جدول المنتجات:')
display(df_products)

# عرض جدول العملاء
df_customers = pd.read_sql_query('SELECT * FROM customers', conn)
print('\nجدول العملاء:')
display(df_customers)

# عرض جدول الطلبات
df_orders = pd.read_sql_query('SELECT * FROM orders', conn)
print('\nجدول الطلبات:')
display(df_orders)

## 1. إدراج منتج جديد (INSERT)
أضف منتجًا جديدًا إلى جدول products باستخدام جملة INSERT. استخدم الاسم 'Headphones' والفئة 'Accessories' والسعر 120.0 والمخزون 30.

In [ ]:
# TODO: اكتب جملة INSERT لإضافة المنتج الجديد (الاسم: 'Headphones', الفئة: 'Accessories', السعر: 120.0, المخزون: 30)

conn.commit()
# عرض المنتجات بعد الإضافة
df = pd.read_sql_query('SELECT * FROM products', conn)
display(df)

## 2. تحديث الأسعار (UPDATE)
قم بزيادة سعر جميع المنتجات في فئة 'Electronics' بنسبة 10%.

In [ ]:
# TODO: اكتب جملة UPDATE لزيادة السعر بنسبة 10% لمنتجات فئة 'Electronics'

conn.commit()
# عرض المنتجات بعد التحديث
df = pd.read_sql_query('SELECT * FROM products', conn)
display(df)

## 3. حذف الطلبات القديمة (DELETE)
احذف جميع الطلبات التي تمت قبل تاريخ '2023-01-01'.

In [ ]:
# TODO: اكتب جملة DELETE لحذف الطلبات التي order_date < '2023-01-01'

conn.commit()
# عرض الطلبات بعد الحذف
df = pd.read_sql_query('SELECT * FROM orders', conn)
display(df)

## 4. المعاملات (Transactions)
تسمح المعاملات بتنفيذ مجموعة من العمليات كوحدة واحدة. إذا فشلت أي عملية، يمكن التراجع عن الكل (ROLLBACK) وإلا يتم الالتزام (COMMIT).
في هذا التمرين، سنقوم بإدراج عميل جديد وطلب له في معاملة واحدة.

In [ ]:
# TODO: املأ الفراغات لاستخدام معاملة بشكل صحيح
try:
    # TODO: ابدأ المعاملة
    
    # إدراج عميل جديد
    cursor.execute("INSERT INTO customers (name, email, join_date) VALUES ('Khaled', 'khaled@example.com', '2023-06-15')")
    # الحصول على معرف العميل الجديد
    new_customer_id = cursor.lastrowid
    # إدراج طلب جديد لهذا العميل
    cursor.execute("INSERT INTO orders (customer_id, product_id, quantity, order_date) VALUES (?, 1, 2, '2023-06-15')", (new_customer_id,))
    # TODO: إذا نجحت العمليات، قم بالالتزام (COMMIT)
    print('تمت العملية بنجاح وتم الالتزام.')
except sqlite3.Error as e:
    # TODO: في حالة الخطأ، قم بالتراجع (ROLLBACK)
    print('حدث خطأ:', e)
    print('تم التراجع عن جميع العمليات.')
finally:
    # عرض النتائج
    print('\nالعملاء بعد العملية:')
    print(pd.read_sql_query('SELECT * FROM customers', conn))
    print('\nالطلبات بعد العملية:')
    print(pd.read_sql_query('SELECT * FROM orders', conn))

## 5. مشروع: سيناريو تعديل متكامل
ستقوم بتنفيذ سيناريو كامل: إضافة عميل جديد، ثم طلب له، ثم تعديل كمية الطلب، ثم حذف طلب إذا كان تاريخه قبل تاريخ معين. استخدم معاملة واحدة لضمان التناسق.

In [ ]:
# TODO: ابدأ معاملة جديدة

try:
    # 1. إدراج عميل جديد (الاسم: 'Mona', البريد: 'mona@example.com', تاريخ الانضمام: '2023-07-01')
    # TODO: اكتب جملة INSERT لإدراج العميل
    
    # احصل على معرّف العميل الجديد
    customer_id = cursor.lastrowid
    
    # 2. إدراج طلب جديد لهذا العميل (المنتج بمعرف 2، الكمية 5، تاريخ الطلب '2023-07-10')
    # TODO: اكتب جملة INSERT لإدراج الطلب
    
    # 3. تعديل كمية الطلب: زيادة الكمية إلى 10
    # TODO: اكتب جملة UPDATE لتعديل كمية الطلب الذي أضفته للتو إلى 10
    # (تلميح: استخدم lastrowid بعد إدراج الطلب للحصول على order_id)
    
    # 4. حذف الطلب إذا كان تاريخه قبل '2023-07-01' (لن ينطبق هنا لكن للتوضيح)
    # TODO: اكتب جملة DELETE لحذف الطلب إذا كان order_date < '2023-07-01'
    
    # إذا وصلنا إلى هنا، قم بتنفيذ الالتزام
    # TODO: أضف COMMIT
    print('تم تنفيذ السيناريو بنجاح')
except sqlite3.Error as e:
    # TODO: في حالة الخطأ، قم بالتراجع ROLLBACK
    print('فشل السيناريو، تم التراجع:', e)
finally:
    # عرض النتائج
    print('\nالعملاء:')
    print(pd.read_sql_query('SELECT * FROM customers', conn))
    print('\nالطلبات:')
    print(pd.read_sql_query('SELECT * FROM orders', conn))

## خلاصة
في هذا المختبر تدربنا على أوامر INSERT، UPDATE، DELETE واستخدام المعاملات لضمان تناسق البيانات. يمكنك تجربة المزيد من السيناريوهات.